# 02 - Built-in On-Demand Evaluations

This notebook evaluates existing AgentCore traces without redeploying the agent.

You will:

1. Choose evaluators by target level and failure mode.
2. Run a targeted evaluation with the AgentCore CLI.
3. Automate the same workflow with `EvaluationClient`.
4. Interpret value, label, and explanation without assuming universal thresholds.

**Estimated time:** 30-45 minutes  
**Creates AWS resources:** No persistent resources; evaluation calls incur model usage.

## 1. Current built-in catalog

AgentCore CLI `0.27.0` recognizes these built-in evaluator IDs:

| Evaluator | Level | Use it to ask |
|---|---|---|
| `Builtin.GoalSuccessRate` | SESSION | Did the conversation complete the user's goal? |
| `Builtin.Correctness` | TRACE | Is the response correct, especially against a reference? |
| `Builtin.Faithfulness` | TRACE | Is the response grounded in supplied context? |
| `Builtin.Helpfulness` | TRACE | Is the response useful? |
| `Builtin.ResponseRelevance` | TRACE | Does the response address the request? |
| `Builtin.Conciseness` | TRACE | Is the response appropriately concise? |
| `Builtin.Coherence` | TRACE | Is the response logically coherent? |
| `Builtin.InstructionFollowing` | TRACE | Did the agent follow its instructions? |
| `Builtin.Refusal` | TRACE | Did the agent handle refusal behavior appropriately? |
| `Builtin.ToolSelectionAccuracy` | TOOL_CALL | Was this the right tool for the task? |

Verify the available evaluator IDs for your installed version with `agentcore run eval --help`, because the built-in catalog can evolve between CLI releases.

## 2. Setup and create a fresh session

On-demand evaluation scores spans that already exist. It does not invoke the agent for you unless you use the dataset option covered in notebook 03.

In [ ]:
import json
from pathlib import Path

from src.workshop_utils import (
    GENERATED_DIR,
    RUNTIME_NAME,
    make_session_id,
    run_cli_json,
    wait_for_session_trace,
)

GENERATED_DIR.mkdir(exist_ok=True)
SESSION_ID = make_session_id("builtins")
PROMPT = "What are the population and land area of Seattle, WA?"

invocation = run_cli_json(
    "invoke",
    "--runtime",
    RUNTIME_NAME,
    "--session-id",
    SESSION_ID,
    "--prompt",
    PROMPT,
)
print(json.dumps(invocation, indent=2))
trace = wait_for_session_trace(SESSION_ID)
trace

## 3. CLI first: one evaluation command

Ground truth can include:

- an `expected_response` for semantic comparison
- one or more natural-language `assertions`
- an `expected_trajectory` listing intended tool names

These are reference inputs, not string-equality rules. An evaluator uses the reference information appropriate to its level and rubric.

In [ ]:
cli_result = run_cli_json(
    "run",
    "eval",
    "--runtime",
    RUNTIME_NAME,
    "--session-id",
    SESSION_ID,
    "--evaluator",
    "Builtin.GoalSuccessRate",
    "Builtin.Correctness",
    "Builtin.Helpfulness",
    "Builtin.InstructionFollowing",
    "Builtin.ToolSelectionAccuracy",
    "--expected-response",
    (
        "Seattle, WA has population 780995 and land area "
        "83.8 square miles in the workshop dataset."
    ),
    "--assertion",
    "The response must identify Seattle, WA and use the workshop facts.",
    "--expected-trajectory",
    "lookup_city",
)
print(json.dumps(cli_result, indent=2))

The CLI saves run history under `agentcore/.cli/eval-runs/` and exposes it through:

```bash
agentcore evals history --limit 10
```

Use the CLI when you are investigating a recent session, iterating on an evaluator, or need a simple CI command against existing traces.

In [ ]:
history = run_cli_json("evals", "history", "--limit", "10")
history

## 4. Python automation with `EvaluationClient`

`EvaluationClient`:

1. queries CloudWatch for a session's spans
2. looks up each evaluator's level
3. creates the correct session, trace, or span targets
4. calls the AgentCore `Evaluate` API

It is useful when evaluation is part of a notebook, test harness, or Python pipeline.

In [ ]:
from datetime import timedelta

from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs
from src.workshop_utils import load_runtime_info

runtime = load_runtime_info()
evaluation_client = EvaluationClient(region_name=runtime.region)

references = ReferenceInputs(
    expected_response=(
        "Seattle, WA has population 780995 and land area "
        "83.8 square miles in the workshop dataset."
    ),
    assertions=[
        "The answer identifies Seattle, WA.",
        "The answer uses the fixed workshop facts.",
    ],
    expected_trajectory=["lookup_city"],
)

sdk_results = evaluation_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.Helpfulness",
        "Builtin.ToolSelectionAccuracy",
    ],
    session_id=SESSION_ID,
    agent_id=runtime.runtime_id,
    look_back_time=timedelta(hours=1),
    reference_inputs=references,
)
sdk_results

## 5. Normalize the response for analysis

Do not code against one display layout. Preserve the raw response and select the fields you need.

In [ ]:
import pandas as pd

rows = []
for item in sdk_results:
    rows.append(
        {
            "evaluator": item.get("evaluatorId"),
            "value": item.get("value"),
            "label": item.get("label"),
            "explanation": item.get("explanation"),
            "error_code": item.get("errorCode"),
            "error_message": item.get("errorMessage"),
        }
    )

results_df = pd.DataFrame(rows)
results_df

## 6. Interpret results responsibly

`value`, `label`, and `explanation` answer different questions:

- **value** supports aggregation and comparison
- **label** provides the evaluator's categorical interpretation
- **explanation** helps diagnose the observed behavior

There is no universal production threshold such as "0.8 is always good." Thresholds depend on the evaluator, traffic distribution, judge calibration, and the cost of false positives and false negatives.

A defensible threshold is established by:

1. collecting representative human-labeled examples
2. running the evaluator on those examples
3. measuring disagreement and repeatability
4. selecting a threshold tied to an operational decision
5. monitoring the threshold after deployment

## 7. Evaluator selection exercise

Choose the smallest set that diagnoses your likely failure:

| Suspected failure | Start with |
|---|---|
| Correct facts, but the task is unfinished | `GoalSuccessRate` |
| Wrong city values | `Correctness` with an expected response |
| Correct answer, wrong or unnecessary tool | `ToolSelectionAccuracy` |
| Long answer that ignores the output contract | `InstructionFollowing` and a custom schema evaluator |
| Unsupported claims after retrieval | `Faithfulness` |

Running every evaluator on every trace increases cost and often produces a noisy dashboard. Evaluation design should follow the failure modes you intend to detect.

## 8. Checkpoint

You now have two equivalent interfaces:

- CLI for fast, targeted inspection
- Python for reusable evaluation automation

Continue to [03 - Ground Truth and Curated Datasets](03-ground-truth-and-datasets.ipynb) to turn one-session inspection into a repeatable regression suite.